## Connect

In [1]:
import duckdb, requests, zipfile
import pandas as pd
from pathlib import Path

BASE = Path.home() / "Desktop/projects/claims_analysis"
con = duckdb.connect(str(BASE / "data/claims.duckdb"))

## Download ICD-9 code names from CMS

In [2]:
LOOK = BASE / "data/lookups"
LOOK.mkdir(exist_ok=True)

url = "https://www.cms.gov/Medicare/Coding/ICD9ProviderDiagnosticCodes/Downloads/ICD-9-CM-v32-master-descriptions.zip"
z = LOOK / "icd9_v32.zip"
if not z.exists():
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    z.write_bytes(r.content)

zipfile.ZipFile(z).extractall(LOOK)
sorted(p.name for p in LOOK.iterdir())

['CMS32_DESC_LONG_DX.txt',
 'CMS32_DESC_LONG_SG.txt',
 'CMS32_DESC_LONG_SHORT_DX.xlsx',
 'CMS32_DESC_LONG_SHORT_SG.xlsx',
 'CMS32_DESC_SHORT_DX.txt',
 'CMS32_DESC_SHORT_SG.txt',
 'icd9_v32.zip']

## Load the code names into DuckDB

In [3]:
def read_codes(pattern):
    f = next(LOOK.glob(pattern))
    rows = [line.strip().split(maxsplit=1) for line in open(f, encoding="latin-1") if line.strip()]
    return pd.DataFrame(rows, columns=["code", "description"])

icd9_dx = read_codes("*LONG_DX*")
icd9_proc = read_codes("*LONG_SG*")

con.execute("CREATE OR REPLACE TABLE icd9_dx AS SELECT * FROM icd9_dx")
con.execute("CREATE OR REPLACE TABLE icd9_proc AS SELECT * FROM icd9_proc")
len(icd9_dx), len(icd9_proc)

(14567, 3882)

## Top reasons for hospital stays (ICD-9 diagnoses)

In [4]:
con.sql("""
SELECT s.principal_dx AS icd9, d.description,
    COUNT(*) AS stays,
    ROUND(AVG(s.paid), 0) AS avg_paid,
    ROUND(SUM(s.paid) / 1e6, 2) AS total_paid_millions
FROM stays s LEFT JOIN icd9_dx d ON s.principal_dx = d.code
GROUP BY ALL ORDER BY stays DESC LIMIT 15
""").df()

,icd9,description,stays,avg_paid,total_paid_millions
0,486,"Pneumonia, organism unspecified",2508,6991.0,17.53
1,V5789,Care involving other specified rehabilitation ...,1761,15900.0,28.00
2,0389,Unspecified septicemia,1660,14793.0,24.56
3,41401,Coronary atherosclerosis of native coronary ar...,1641,15244.0,25.02
4,49121,Obstructive chronic bronchitis with (acute) ex...,1582,6184.0,9.78
5,4280,"Congestive heart failure, unspecified",1481,8011.0,11.86
6,5990,"Urinary tract infection, site not specified",1463,5398.0,7.90
7,42731,Atrial fibrillation,1266,6131.0,7.76
8,5849,"Acute kidney failure, unspecified",1163,8120.0,9.44
9,41071,"Subendocardial infarction, initial episode of ...",1162,12842.0,14.92


## Most expensive DRGs

In [11]:
con.sql("""
SELECT drg, COUNT(*) AS stays, ROUND(AVG(paid), 0) AS avg_paid,
    ROUND(SUM(paid) / 1e6, 2) AS total_paid_millions
FROM stays GROUP BY drg ORDER BY total_paid_millions DESC LIMIT 15
""").df()

,drg,stays
0,198,76
1,175,72
2,176,72
3,187,71
4,180,70
5,208,68
6,165,67
7,201,66
8,181,66
9,168,65


## Top inpatient procedures (ICD-9)

In [8]:
con.sql("""
SELECT i.ICD9_PRCDR_CD_1 AS icd9_proc, p.description, COUNT(*) AS claims
FROM inpatient i LEFT JOIN icd9_proc p ON i.ICD9_PRCDR_CD_1 = p.code
WHERE i.ICD9_PRCDR_CD_1 IS NOT NULL
GROUP BY ALL ORDER BY claims DESC LIMIT 15
""").df()

,icd9_proc,description,claims
0,9904,Transfusion of packed cells,1881
1,8154,Total knee replacement,1608
2,3893,"Venous catheterization, not elsewhere classified",1473
3,3995,Hemodialysis,1336
4,0066,Percutaneous transluminal coronary angioplasty...,1329
5,4516,Esophagogastroduodenoscopy [EGD] with closed b...,1038
6,3722,Left heart cardiac catheterization,997
7,9671,Continuous invasive mechanical ventilation for...,750
8,8872,Diagnostic ultrasound of heart,722
9,8151,Total hip replacement,720


## Top doctor services (CPT/HCPCS)

In [9]:
lines = " UNION ALL ".join(
    f"SELECT DESYNPUF_ID, year(CLM_FROM_DT) AS year, HCPCS_CD_{i} AS hcpcs, LINE_NCH_PMT_AMT_{i} AS paid "
    f"FROM carrier WHERE HCPCS_CD_{i} IS NOT NULL"
    for i in range(1, 14))
con.execute(f"CREATE OR REPLACE TABLE carrier_lines AS {lines}")

con.sql("""
SELECT hcpcs, COUNT(*) AS services, ROUND(SUM(paid) / 1e6, 2) AS paid_millions,
    ROUND(AVG(paid), 0) AS avg_paid
FROM carrier_lines GROUP BY hcpcs ORDER BY paid_millions DESC LIMIT 15
""").df()

,hcpcs,services,paid_millions,avg_paid
0,99213,599911,32.10,54.0
1,99214,429412,23.08,54.0
2,99232,252762,15.73,62.0
3,36415,367196,9.03,25.0
4,99233,109230,7.31,67.0
5,99212,118062,6.52,55.0
6,93010,115553,5.92,51.0
7,92014,63361,5.79,91.0
8,88305,70912,5.56,78.0
9,71010,115348,5.51,48.0


In [12]:
con.sql("""
SELECT drg, COUNT(*) AS stays
FROM stays WHERE principal_dx = '486'
GROUP BY drg ORDER BY stays DESC LIMIT 10
""").df()

,drg,stays
0,198,76
1,175,72
2,176,72
3,187,71
4,180,70
5,208,68
6,165,67
7,181,66
8,201,66
9,183,65


In [10]:
con.execute("CALL start_ui()").fetchall()

[('UI started at http://localhost:4213/',)]